# Single-phase NLF as a two-tap FIR

Load real Q1.15 taps +0.5 and −0.25. With OUT_SCALE=0 the expected output codes are 4 x[n] − 2 x[n−1] for both I and Q. Measure the common pipeline delay from the impulse peak and apply it unchanged to a separate coherent complex tone. Require exact code agreement, 1024 valid samples, zero stream faults and zero overflow flags on each frame.

In [ ]:
import json, os, sys
from pathlib import Path
import numpy as np
from pynq import Overlay
import pynq_dfe
import matplotlib.pyplot as plt

root = Path(os.environ.get("DFE_REPO", "..")).resolve()
example = root / "packages/pynq-dfe/examples/nlf"
sys.path[:0] = [str(example), str(root / "overlay/dfe-resampler-nlf")]
from capture import capture
from model import *
bit = Path(os.environ["DFE_BITFILE"])
out = Path(os.environ.get("DFE_OUTPUT", ".")).resolve()
out.mkdir(exist_ok=True)
ol = Overlay(str(bit))
rows, traces = [], {}

def check(name, measured, predicted, passed):
    rows.append(dict(test=name, measured=float(measured), predicted=float(predicted), passed=bool(passed)))

In [ ]:
taps = (16384, -8192)
ol.nlf.load_fir(taps)
print("NLF version and configuration", hex(ol.nlf.read(0x20)), hex(ol.nlf.read(0x24)))
n = np.arange(1024)
tone = 8192 * np.exp(2j * np.pi * 79 * n / 1024)
impulse = np.zeros(1024, complex)
impulse[0] = 8192 + 4096j
for kind, x in (("impulse", impulse), ("tone", np.rint(tone.real)+1j*np.rint(tone.imag))):
    y, beats, faults = capture(ol, 2, x)
    if kind == "impulse":
        delay = int(np.argmax(abs(y)))
        print("Common latency (samples modulo 1024)", delay)
    pred = np.roll(predict(x, taps), delay)
    traces.update({kind:y, kind+"_predicted":pred, kind+"_input":x})
    err = np.max(np.abs(y-pred))
    check(kind+" samples", beats, 1024, beats == 1024)
    check(kind+" stream faults", faults, 0, faults == 0)
    over = ol.nlf.read(0x10) & 7
    check(kind+" overflow", over, 0, over == 0)
    check(kind+" maximum error (codes)", err, 0, err == 0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, kind in zip(axes, ("impulse","tone")):
    idx = (delay + np.arange(-4,44)) % 1024
    ax.plot(np.arange(-4,44), traces[kind+"_predicted"].real[idx], label="Predicted")
    ax.plot(np.arange(-4,44), traces[kind].real[idx], ".", label="Measured")
    ax.set(xlabel="Output sample relative to pipeline delay", ylabel=kind+" I (codes)")
    ax.legend()
fig.tight_layout()
fig.savefig(out / "response.png")

In [ ]:
np.savez_compressed(out / "captures.npz", **traces)
(out / "measurements.json").write_text(json.dumps(rows, indent=2))
for r in rows:
    print(f"{r['test']:38s} {r['measured']:12.6g} {r['predicted']:12.6g}  {'PASS' if r['passed'] else 'FAIL'}")
assert all(r["passed"] for r in rows)